# Classification: Oil Spill Detection (Imbalanced Data)

**Dataset:** `oil_spill.csv`

**Objective:** Build a classifier for a highly imbalanced dataset (~5% minority class) using SMOTE and an imbalanced-friendly pipeline.

**Key Concepts:**
- **Imbalanced Data:** Why accuracy is misleading when one class dominates.
- **SMOTE:** Synthetic Minority Over-sampling Technique.
- **ImbPipeline:** A specialized pipeline from `imblearn` that correctly handles resampling during cross-validation (only applying it to training folds).

---

### Step 1: Setup & Data Loading

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline

# Define column names (49 radar features + 1 target)
explainable_features = ['patch_id', 'area', 'perimeter', 'circularity', 'compactness', 'is_complex_shape', 'edge_gradient_mean', 'edge_gradient_std', 'contrast_local', 'contrast_global']
radar_features = [f'radar_{i}' for i in range(11, 49)]
all_columns = explainable_features + radar_features + ['target']

# Load dataset
df = pd.read_csv('../data/raw/oil_spill.csv', header=None, names=all_columns)

print(f"Dataset Shape: {df.shape}")
df.head()

### Step 2: Analyze Class Imbalance

We visualize the target distribution to understand the scale of the imbalance.

In [ ]:
plt.figure(figsize=(6, 4))
sns.countplot(data=df, x='target', palette='Set1')
plt.title('Distribution of Oil Spills')
plt.show()

print("Class Distribution Ratios:")
print(df['target'].value_counts(normalize=True).round(3))

### Step 3: Train-Test Split (Stratified)

We must use `stratify=y` to ensure the 5% minority class is preserved in both splits.

In [ ]:
X = df.drop(columns=['target'])
y = df['target']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

print(f"Training set: {X_train.shape}")
print(f"Testing set:  {X_test.shape}")

### Step 4: Build & Train the SMOTE Pipeline

We use `ImbPipeline` so that SMOTE is **only** applied to the training data during `fit()`, and skipped during `predict()` on the test set.

In [ ]:
model_pipeline = ImbPipeline(steps=[
    ('scaler', StandardScaler()),
    ('smote', SMOTE(random_state=42)),
    ('classifier', RandomForestClassifier(random_state=42, n_estimators=100))
])

# Train
model_pipeline.fit(X_train, y_train)

# Evaluate
y_pred = model_pipeline.predict(X_test)

print(f"Overall Accuracy: {accuracy_score(y_test, y_pred):.4f}\n")
print("Classification Report (Focus on Class 1 Recall/F1):")
print(classification_report(y_test, y_pred))